# Varlen Transformer (SM75 Turing / Tesla T4) — Build, Verify, Benchmark & NCU Profile

Notebook hướng dẫn đầy đủ trên GPU **NVIDIA Tesla T4 (SM75)** (Google Colab / Kaggle / máy GPU cục bộ):
1. Kiểm tra môi trường GPU, CUDA, PyTorch và chuẩn bị thư viện **NVIDIA CUTLASS**.
2. Biên dịch CUDA/CuTe extension và đóng gói file `.whl` vào thư mục `wheels/` để tái sử dụng không cần biên dịch lại.
3. Kiểm tra độ chính xác toán học (**Forward**, **Backward**, **Flash Decoding**, và **torch.compile / TorchInductor**).
4. Chạy **Training Benchmark** (Varlen CUDA vs Batched SDPA Padded) và lưu biểu đồ vào `assets/training_benchmark.png`.
5. Chạy **Inference Benchmark** (Split-K Decode `Lq=1` + E2E Prefill/Generation) và lưu biểu đồ vào `assets/inference_benchmark.png`.
6. Chạy **NVIDIA Nsight Compute (`ncu`)** xuất báo cáo `.ncu-rep` cho 3 kernel vào thư mục `Benchmark/`.

## 1. Kiểm tra GPU & Chuẩn bị Mã nguồn / CUTLASS

In [ ]:
import os, sys, subprocess

if not os.path.isfile("setup.py") and not os.path.isdir("Varlen-Attention"):
    !git clone --recursive --depth 1 https://github.com/CBG6682/Varlen-Attention.git

if os.path.isdir("Varlen-Attention"):
    %cd Varlen-Attention
elif os.path.basename(os.getcwd()) == "Benchmark":
    %cd ..

print("Working directory:", os.getcwd())

import torch
print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available(), "| CUDA runtime:", torch.version.cuda)
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0), "| Capability:", torch.cuda.get_device_capability(0))

!nvidia-smi

In [ ]:
%%bash
set -e
if [ ! -f "third_party/cutlass/include/cute/tensor.hpp" ]; then
    echo "Initializing NVIDIA CUTLASS submodule in third_party/cutlass..."
    git submodule update --init --depth 1 third_party/cutlass
else
    echo "CUTLASS headers already exist in third_party/cutlass/include."
fi

## 2. Biên dịch Extension & Đóng gói File Wheel (`wheels/`)
Nếu đã có sẵn file `.whl` khớp phiên bản Python/PyTorch trong `wheels/`, có thể cài đặt trực tiếp không cần biên dịch lại.

In [ ]:
%%bash
set -euo pipefail
export MAX_JOBS=2
export FORCE_REBUILD=1
export VERBOSE=1
export TORCH_EXTENSIONS_VERBOSE=1

pip install -q ninja wheel setuptools
# Biên dịch extension (C++20, tự fallback C++17 nếu lỗi) + in đầy đủ log lỗi nvcc/cxx (-v)
python setup.py build_ext --inplace -v
# Đóng gói file wheel (.whl) chứa sẵn nhị phân _C*.so vào thư mục wheels/
FORCE_REBUILD=0 python -m pip wheel -v --no-deps --no-build-isolation -w wheels .
# Cài đặt package dạng editable trỏ tới mã nguồn hiện tại
FORCE_REBUILD=0 python -m pip install -v --no-deps --no-build-isolation -e .
ls -lh wheels/*.whl varlen_transformer/_C*.so

## 3. Kiểm tra Độ chính xác (Correctness + Autograd + torch.compile)
So sánh từng kernel (`mha_fwd`, `mha_bwd`, `mha_decode`), module GQA đầu-cuối với `VarlenKVCache`, và chế độ `torch.compile(backend='inductor')`.

In [ ]:
!python tests/test_correctness.py

## 4. Training Benchmark (Forward & Forward + Backward vs torch.compile Baseline Speed)
Đo tốc độ huấn luyện lấy **BatchedFlashGQABaselineSpeed + torch.compile** (TorchInductor) làm baseline tốc độ chuẩn, đối đầu trực tiếp với custom kernel **VarlenFlashGQA + torch.compile** (nhờ cơ chế FakeTensor đã đăng ký) và chế độ Eager.

In [ ]:
!python tests/bench_training.py \
    --batch-sizes 4 8 16 \
    --max-lens 512 1024 2048 \
    --distributions skewed uniform \
    --save-plot assets/training_benchmark.png

In [ ]:
from IPython.display import Image, display
import os
if os.path.exists("assets/training_benchmark.png"):
    display(Image("assets/training_benchmark.png"))

## 5. Inference Benchmark (Decode `Lq=1` & End-to-End: Baseline vs Varlen Auto-Dispatch)
Lấy **torch.compile (`SDPBackend.EFFICIENT_ATTENTION` / xFormers CUTLASS)** làm baseline tốc độ chuẩn với cấu hình lớn (`H_q=32, H_kv=8`, batch mở rộng lên tới `B=64/96` ở `L` ngắn để tận dụng tối đa VRAM và giữ bận toàn bộ SM). Part A đo độ trễ/băng thông `Lq=1`; Part B (E2E) đo **Baseline** vs **Varlen Auto-Dispatch** (tự động chọn Direct 1-Block hoặc Split-K theo `B * H_q` và occupancy) cùng biểu đồ **E2E Peak VRAM**.

> **Lưu ý kỹ thuật về Baseline SDPA trên T4 (`SM75`):** PyTorch SDPA với `pad_mask` kiểu `bool` sẽ tự động rơi về nhân `MATH` (chậm). Do đó, benchmark đã chuyển `pad_mask` sang **additive `float16` (`0.0` / `-inf`)**, căn chỉnh chiều cuối `L` là **bội số của 8** với stride tường minh `(aligned_len, 0, 0, 1)` để ép kích hoạt nhân **`SDPBackend.EFFICIENT_ATTENTION`** (xFormers) công bằng nhất cho baseline.

In [ ]:
# Thêm cờ --e2e-quick nếu muốn chạy nhanh tập con 5 cấu hình
!python tests/bench_inference.py \
    --batch-sizes 1 8 32 64 \
    --max-lens 128 256 1024 2048 4096 \
    --num-heads-q 32 \
    --num-heads-kv 8 \
    --distributions skewed uniform \
    --gen-steps 32 \
    --save-plot assets/inference_benchmark.png

In [ ]:
from IPython.display import Image, display
import os
if os.path.exists("assets/inference_benchmark.png"):
    display(Image("assets/inference_benchmark.png"))

## 6. Nsight Compute Profiling (`.ncu-rep` trong `Benchmark/`)
Thu thập hồ sơ hiệu năng phần cứng (`SM Throughput`, `DRAM Bandwidth`, `Achieved Occupancy`, `Register/Thread`) và **nhúng trực tiếp mã nguồn CUDA/C++ (`--import-source yes`)** vào 3 file báo cáo:
- `Benchmark/mha_fwd.ncu-rep` (chạy 4 kịch bản: `short_b1`, `short_b16`, `skewed_b4`, `long_b1`)
- `Benchmark/mha_bwd.ncu-rep` (chạy 4 kịch bản: `short_b1`, `short_b16`, `skewed_b4`, `long_b1`)
- `Benchmark/mha_decode.ncu-rep` (chạy cả 4 kịch bản + đối chiếu trực tiếp nhánh Direct 1-Block vs Split-K trên `skewed_b4`)

> Lưu ý: Trên một số container cloud không cấp quyền hardware counter cho `ncu`, lệnh có thể báo `ERR_NVGPUCTRPERM`.

In [ ]:
%%bash
set -e
if ! command -v ncu &> /dev/null; then
    echo "[INFO] NVIDIA Nsight Compute (ncu) not found in PATH."
    exit 0
fi

NCU_SCRIPT='import sys, torch, varlen_transformer as vt
mode = sys.argv[1]
dev = torch.device("cuda")
cases = [
    ("short_b1", [128]),
    ("short_b16", [128] * 16),
    ("skewed_b4", [256, 512, 1024, 2048]),
    ("long_b1", [4096]),
]
rope = vt.RotaryPositionalEmbedding(64, max_seq_len=4200, device=dev)
cos, sin = rope.get_cos_sin(device=dev, dtype=torch.float16)
for cname, lens in cases:
    cu = torch.tensor([0] + torch.cumsum(torch.tensor(lens), 0).tolist(), dtype=torch.int32, device=dev)
    total_l = int(cu[-1].item())
    max_l = max(lens)
    B = len(lens)
    k = torch.randn((4, total_l, 64), device=dev, dtype=torch.float16)
    v = torch.randn((4, total_l, 64), device=dev, dtype=torch.float16)
    if mode == "fwd":
        q = torch.randn((16, total_l, 64), device=dev, dtype=torch.float16)
        vt.fused_mha_varlen_fwd(q, k, v, cu, cu, max_l, cos=cos, sin=sin, is_causal=True)
    elif mode == "bwd":
        q = torch.randn((16, total_l, 64), device=dev, dtype=torch.float16)
        o, lse = vt.fused_mha_varlen_fwd(q, k, v, cu, cu, max_l, cos=cos, sin=sin, is_causal=True)
        do = torch.randn_like(o)
        vt.fused_mha_varlen_bwd(do, q, k, v, o, lse, cu, cu, None, max_l, cos=cos, sin=sin, is_causal=True)
    elif mode == "decode":
        qd = torch.randn((16, B, 64), device=dev, dtype=torch.float16)
        vt.fused_mha_varlen_decode(qd, k, v, cu, max_l, cos=cos, sin=sin)
        if cname == "skewed_b4":
            vt.fused_mha_varlen_decode(qd, k, v, cu, max_l, cos=cos, sin=sin, use_splitk=False)
            vt.fused_mha_varlen_decode(qd, k, v, cu, max_l, cos=cos, sin=sin, use_splitk=True)
torch.cuda.synchronize()'

echo "$NCU_SCRIPT" > Benchmark/_ncu_profile_target.py

for MODE in fwd bwd decode; do
    echo "=== Profiling mha_${MODE} (multi-case + embedded source) ==="
    ncu --set full --import-source yes --source-folders .,src,include -k "regex:kernel_fused_mha_varlen_${MODE}" -f -o "Benchmark/mha_${MODE}" python Benchmark/_ncu_profile_target.py "${MODE}" || true
done

rm -f Benchmark/_ncu_profile_target.py
ls -lh Benchmark/*.ncu-rep 2>/dev/null || echo "No .ncu-rep files generated."